# Customer Support Ticket Router

A small LangGraph example that classifies support tickets with keyword rules and routes them to a billing or technical response node. It uses no LLM or API key.

## Install dependencies

Run this setup cell in Google Colab before running the example.

In [ ]:
%pip install -q langgraph

## Define the graph and ticket state

In [ ]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class TicketState(TypedDict):
    ticket: str
    category: str
    reply: str


def classify_ticket(state: TicketState) -> dict[str, str]:
    text = state["ticket"].lower()
    billing_keywords = ("bill", "charge", "payment", "refund", "invoice", "subscription")
    category = "billing" if any(word in text for word in billing_keywords) else "technical"
    return {"category": category}


def route_ticket(state: TicketState) -> Literal["billing", "technical"]:
    return state["category"]  # Conditional edge selects the matching handler.


def handle_billing(state: TicketState) -> dict[str, str]:
    return {"reply": "I can help with your billing question. Please share the invoice or charge details."}


def handle_technical(state: TicketState) -> dict[str, str]:
    return {"reply": "I can help troubleshoot this technical issue. Please share any error message and the steps to reproduce it."}


builder = StateGraph(TicketState)
builder.add_node("classify", classify_ticket)
builder.add_node("billing", handle_billing)
builder.add_node("technical", handle_technical)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route_ticket)
builder.add_edge("billing", END)
builder.add_edge("technical", END)

ticket_router = builder.compile()

## Draw the graph

In [ ]:
from IPython.display import Image, display

display(Image(ticket_router.get_graph().draw_mermaid_png()))

## Try one ticket on each route

In [ ]:
sample_tickets = [
    "I was charged twice for my monthly subscription.",
    "The app crashes with an error when I try to upload a file.",
]

for ticket in sample_tickets:
    result = ticket_router.invoke({"ticket": ticket, "category": "", "reply": ""})
    print(f"Ticket: {result['ticket']}")
    print(f"Category: {result['category']}")
    print(f"Reply: {result['reply']}\n")